In [ ]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
import scipy.stats as sps

In [ ]:
PAIR = 'BTC_USDT'
TIMEFRAME = '1d'

EXCHANGE = 'binance'

odf = pd.read_json(f'/media/mu6mula/Data/Crypto-Data-Feed/freq-user-data/data/{EXCHANGE}/{PAIR}-{TIMEFRAME}.json'
# exchange = 'kucoin'
# odf = pd.read_json(f'../../freq-user-data/data/{exchange}/futures/{pair}-{timeframe}-futures.json'
).dropna().set_axis(['timestamp', 'open', 'high', 'low', 'close', 'volume'], axis=1
).assign(dtime=lambda x: pd.to_datetime(x['timestamp'], unit='ms', utc=False)
).drop('timestamp', axis=1
).set_index('dtime')#.sort_index()


In [ ]:
prices = odf.close.values[1:]
raw_data = odf.close
returns = odf.close.pct_change()[1:]
ws, ww = 0,32
prices = prices[ws:ws+ww]
returns = returns[ws:ws+ww]
prices.shape, returns.shape

In [ ]:
returns

In [ ]:

df

In [ ]:

ticker = '^GSPC' 
start = '2015-12-31'
end = '2024-07-16'
#downloading data
# df = yf.download(ticker, start, end)
# df.to_csv('SP500.csv')
df = pd.read_csv('SP500.csv', index_col=0)
raw_data = df['Close']
prices = np.array(raw_data)[1:]
returns = np.array(raw_data)[1:]/np.array(raw_data)[:-1] - 1

In [43]:
#specifying the maximum power of 2
power = 5
#the rolling sample length
n = 2**power
#initialising arrays
hursts = np.array([])
tstats = np.array([])
pvalues = np.array([])
#calculating the rolling Hurst exponent
for t in np.arange(n,len(returns)+1):
    #specifying the subsample
    data = returns[t-n:t]
    X = np.arange(2, power+1)
    Y = np.array([])
    for p in X:
        print('------------')
        m = 2**p
        s = 2**(power-p)
        print(f'p={p},m={m},s={s}')
        rs_array = np.array([])
        #moving across subsamples
        for i in np.arange(0,s):
            subsample = data[i*m:(i+1)*m]
            mean = np.average(subsample)
            deviate = np.cumsum(subsample-mean)
            difference = max(deviate) - min(deviate)
            stdev = np.std(subsample)
            rescaled_range = difference/stdev
            rs_array = np.append(rs_array, rescaled_range)
        #calculating the log2 of average rescaled range
        Y = np.append(Y, np.log2(np.average(rs_array)))
        print(rs_array)
        print('---\n',np.average(rs_array))
    reg = sm.OLS(Y, sm.add_constant(X))
    res = reg.fit()
    hurst = res.params[1]
    tstat = (res.params[1]-0.5)/res.bse[1]
    pvalue = 2*(1 - sps.t.cdf(abs(tstat),res.df_resid))
    hursts = np.append(hursts, hurst)
    tstats = np.append(tstats, tstat)
    pvalues = np.append(pvalues, pvalue)


------------
p=2,m=4,s=8
[1.55564617 1.96602369 1.43121969 1.64125505 1.65411355 1.56379137
 1.62088242 1.50575569]
------------
p=3,m=8,s=4
[3.07235702 2.05600857 1.78733369 3.30881662]
------------
p=4,m=16,s=2
[2.6463394  5.41599336]
------------
p=5,m=32,s=1
[5.04494621]


In [ ]:
#visualising the Hurst exponent
plt.figure(1)
plt.rc('xtick',labelsize = 8)
plt.ylim(0.4,0.6)
plt.plot(raw_data.index[n:],hursts)
plt.plot(raw_data.index[n:],np.ones(len(hursts))*0.5)
plt.show()
#visualising the t-stat and critical values
plt.figure(2)
plt.rc('xtick',labelsize = 8)
plt.plot(raw_data.index[n:],tstats)
plt.plot(raw_data.index[n:],np.ones(len(tstats))*sps.t.ppf(0.005,res.df_resid))
plt.plot(raw_data.index[n:],np.ones(len(tstats))*sps.t.ppf(0.995,res.df_resid))
plt.show()